In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from azure.cosmos import CosmosClient

URL = "https://aegiscosmosdb.documents.azure.com"
KEY = "6Di3nSl2nSG7qitzD8SGYeXuw5rn3eUmC4fha2YswtM5R5hSet8u0hYzMMSnhVBmTfNgMJMxKAbTACDbSPsJpg==" 
DATABASE_NAME = "aegisraw"
CONTAINER_NAME = "openalex-authors"

In [ ]:
# ==========================================
# Fetch Authors Data from CosmosDB
# ==========================================
print("Connecting to Cosmos DB...")
client = CosmosClient(URL, credential=KEY)
database = client.get_database_client(DATABASE_NAME)
container = database.get_container_client(CONTAINER_NAME)

query = """
SELECT 
    c.display_name, 
    c.works_count, 
    c.cited_by_count 
FROM c 
WHERE IS_DEFINED(c.works_count) AND IS_DEFINED(c.cited_by_count)
"""

print("Executing query... grab a coffee, this might take a minute.")
items = list(container.query_items(query=query, enable_cross_partition_query=True))
df_authors = pd.DataFrame(items)

df_authors['works_count'] = pd.to_numeric(df_authors['works_count'], errors='coerce')
df_authors['cited_by_count'] = pd.to_numeric(df_authors['cited_by_count'], errors='coerce')
df_authors = df_authors[df_authors['works_count'] > 0]

print(f"Success! Loaded {len(df_authors)} authors into memory.")

In [ ]:
# ==========================================
# Fetch Works Data from CosmosDB
# ==========================================
WORKS_CONTAINER_NAME = "openalex-works"
works_container = database.get_container_client(WORKS_CONTAINER_NAME)

# We only select the specific arrays we need to keep the memory footprint tiny
works_query = """
SELECT 
    c.authorships, 
    c.topics 
FROM c 
WHERE ARRAY_LENGTH(c.authorships) > 0 AND ARRAY_LENGTH(c.topics) > 0
"""

print("Executing Works query... fetching authors and topics.")
works_items = list(works_container.query_items(query=works_query, enable_cross_partition_query=True))

print(f"Success! Loaded {len(works_items):,} works into memory.")

In [ ]:
# ==========================================
# Extract Author-Topic Relationships
# ==========================================
print("Flattening JSON and extracting author-topic pairs...")

author_topic_pairs = []

for work in works_items:
    # Safely extract all author IDs from this specific work
    author_ids = [
        a.get('author', {}).get('id') 
        for a in work.get('authorships', []) 
        if a.get('author', {}).get('id')
    ]
    
    # Safely extract all topic IDs from this specific work
    topic_ids = [
        t.get('id') 
        for t in work.get('topics', []) 
        if t.get('id')
    ]
    
    # Link every author on the paper to every topic on the paper
    for a_id in author_ids:
        for t_id in topic_ids:
            author_topic_pairs.append({'author_id': a_id, 'topic_id': t_id})

# Convert to a DataFrame
df_author_topics = pd.DataFrame(author_topic_pairs)

# Group by author and count UNIQUE topics
author_stats_df = df_author_topics.groupby('author_id')['topic_id'].nunique().reset_index(name='unique_topics')

print(f"Success! Mapped {len(df_author_topics):,} total topic tags across {len(author_stats_df):,} unique authors.")

In [ ]:
# ==========================================
# Plot Volume vs. Citation Data 
# ==========================================
plt.figure(figsize=(11, 6))

sns.set_theme(style="whitegrid", context="talk") 

# Filter out 0s just to be absolutely safe with the log-scale math
plot_df = df_authors[(df_authors['works_count'] > 0) & (df_authors['cited_by_count'] > 0)]

ax = sns.scatterplot(
    data=plot_df, 
    x='works_count', 
    y='cited_by_count', 
    alpha=0.1,
    s=40,
    edgecolor=None,
    color='#2A9D8F'
)

plt.xscale('log')
plt.yscale('log')

plt.title('Author Volume vs. Citation Impact', fontsize=20, fontweight='bold', pad=20)
plt.xlabel('Total Publications (Log Scale)', fontsize=16, labelpad=10)
plt.ylabel('Total Citations (Log Scale)', fontsize=16, labelpad=10)

# Clean up grid and borders
sns.despine()
ax.grid(True, which="major", ls="--", alpha=0.5)
ax.grid(True, which="minor", ls=":", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Plot Author Productivity Data 
# ==========================================
plt.figure(figsize=(11, 6))

# Set a modern, clean style with larger fonts for presentations
sns.set_theme(style="whitegrid", context="talk") 

# Filter the data 
filtered_authors = df_authors[(df_authors['works_count'] > 0) & (df_authors['works_count'] <= 50)]

# Calculate the exact counts manually
counts = filtered_authors['works_count'].value_counts().sort_index()

plt.bar(
    counts.index, 
    counts.values, 
    color='#2A9D8F',       
    edgecolor='white',     
    linewidth=0.5,         
    alpha=0.9,
    log=True  
)

plt.title('Author Productivity', fontsize=20, fontweight='bold', pad=20)
plt.xlabel('Number of Published Works', fontsize=16, labelpad=10)
plt.ylabel('Number of Authors (Log Scale)', fontsize=16, labelpad=10)

# Clean up borders and grid
sns.despine(left=True)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', visible=False)

plt.xlim(0, 50) 
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Display Productivit/Impact Data 
# ==========================================
total_authors = len(df_authors)

# --- SLIDE 1 STATS: PRODUCTIVITY ---
one_paper = len(df_authors[df_authors['works_count'] == 1])
ten_plus_papers = len(df_authors[df_authors['works_count'] >= 10])

one_paper_pct = (one_paper / total_authors) * 100
ten_plus_pct = (ten_plus_papers / total_authors) * 100

# --- SLIDE 2 STATS: IMPACT ---
zero_cites = len(df_authors[df_authors['cited_by_count'] == 0])
hundred_plus_cites = len(df_authors[df_authors['cited_by_count'] >= 100])

zero_cites_pct = (zero_cites / total_authors) * 100
hundred_plus_pct = (hundred_plus_cites / total_authors) * 100

# Calculate correlation between volume and impact
correlation = df_authors['works_count'].corr(df_authors['cited_by_count'])

# --- PRINT FORMATTED RESULTS ---
print("--- SLIDE 1: PRODUCTIVITY STATS ---")
print(f"Total Authors Analyzed: {total_authors:,}")
print(f"One-and-Done Authors (1 paper): {one_paper_pct:.1f}% ({one_paper:,} people)")
print(f"Potential SMEs (10+ papers): {ten_plus_pct:.1f}% ({ten_plus_papers:,} people)")
print("-" * 40)
print("--- SLIDE 2: IMPACT STATS ---")
print(f"Unvalidated Authors (0 citations): {zero_cites_pct:.1f}% ({zero_cites:,} people)")
print(f"Highly Validated SMEs (100+ citations): {hundred_plus_pct:.1f}% ({hundred_plus_cites:,} people)")
print(f"Volume-to-Impact Correlation: {correlation:.2f} (1.0 is a perfect match)")

In [ ]:
# ==========================================
# Plot Author Distribution Data 
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(11, 6))

# Match the presentation theme
sns.set_theme(style="whitegrid", context="talk") 

all_topics = author_stats_df['unique_topics']

# Calculate exact counts for the log scale
counts = all_topics.value_counts().sort_index()

# Draw the bars
plt.bar(
    counts.index, 
    counts.values, 
    color='#2A9D8F',       
    edgecolor='white',     
    linewidth=0.5,         
    alpha=0.9,
    log=True  
)

# Titles and Labels
plt.title('Distribution of Unique Topics per Author', fontsize=20, fontweight='bold', pad=20)
plt.xlabel('Number of Unique Topics', fontsize=16, labelpad=10)
plt.ylabel('Number of Authors (Log Scale)', fontsize=16, labelpad=10)

# Clean up grid and borders
sns.despine(left=True)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', visible=False)

plt.tight_layout()
plt.show()

# --- PRINT STATS ---
total_authors = len(all_topics)
specialists = len(all_topics[all_topics <= 5])
specialists_pct = (specialists / total_authors) * 100 if total_authors > 0 else 0

generalists = len(all_topics[all_topics > 20])
generalists_pct = (generalists / total_authors) * 100 if total_authors > 0 else 0

print(f"Total Authors Analyzed: {total_authors:,}")
print(f"Hyper-Specialists (1-5 topics): {specialists_pct:.1f}% ({specialists:,} people)")
print(f"Generalists (>20 topics): {generalists_pct:.1f}% ({generalists:,} people)")